In [2]:
# notebook : detectors_comparison.ipynb — cellule 1 : inventaire de results.jsonl (lecture seule)
import hashlib
import json
import re

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

from src.config import RESULT_DIR

PATH = RESULT_DIR / "results.jsonl"
SKIP_PARAMS = ("device", "seed", "model_meta")   # bruit ou champs volumineux
HEAD_KEYS = ("hash", "detector.name", "dataset.split")   # clés récupérées dans une ligne tronquée


def as_scores(raw) -> np.ndarray:
    """Score binaire : colonne unique, ou probabilité de la classe 1 si deux colonnes."""
    s = np.asarray(raw, dtype=float)
    s = s.reshape(len(s), -1)
    if s.shape[1] == 1:
        return s[:, 0]
    if s.shape[1] == 2:
        return s[:, 1]
    raise ValueError(f"y_scores à {s.shape[1]} colonnes")


def score_metrics(y: np.ndarray, s: np.ndarray) -> dict:
    """Mêmes définitions que les tables précédentes : TPR interpolé, accuracies maximales sur la ROC (seuil oracle)."""
    fpr, tpr, _ = roc_curve(y, s, drop_intermediate=False)
    pos = int(y.sum())
    neg = len(y) - pos
    acc = (tpr * pos + (1 - fpr) * neg) / (pos + neg)
    return {"AUC": float(roc_auc_score(y, s)),
            "TPR@1%": float(np.interp(0.01, fpr, tpr)),
            "TPR@5%": float(np.interp(0.05, fpr, tpr)),
            "acc max": float(acc.max()),
            "bacc max": float(((tpr + 1 - fpr) / 2).max())}


def fingerprint(ids, y: np.ndarray) -> str:
    """Empreinte de l'ensemble évalué, indépendante de l'ordre : deux runs comparables ont la même."""
    pairs = sorted(f"{i}|{int(l)}" for i, l in zip(ids, y))
    return hashlib.sha1("\n".join(pairs).encode("utf-8")).hexdigest()[:10]


def salvage_head(prefix: str) -> dict:
    """Valeurs scalaires lisibles avant le point de rupture d'une ligne tronquée."""
    out = {}
    for k in HEAD_KEYS:
        m = re.search(rf'"{re.escape(k)}":\s*"([^"]*)"', prefix)
        if m:
            out[k] = m.group(1)[:12] if k == "hash" else m.group(1)
    return out


rows, corrupt = [], []
n_lines = 0
with open(PATH, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        n_lines = line_no
        if not line.strip():
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            corrupt.append({"ligne": line_no, "taille (car.)": len(line), "pos. erreur": e.pos,
                            "finit par \\n": line.endswith("\n"), "erreur": e.msg, **salvage_head(line[:e.pos])})
            continue
        params = ", ".join(f"{k.split('detector.params.', 1)[1]}={v}" for k, v in rec.items()
                           if k.startswith("detector.params.") and not any(t in k for t in SKIP_PARAMS))
        row = {"ligne": line_no, "hash": str(rec.get("hash", ""))[:12],
               "détecteur": rec.get("detector.name"), "type": rec.get("detector.type"),
               "split": rec.get("dataset.split"), "subset": rec.get("dataset.subset_size"),
               "seed ds": rec.get("dataset.seed"), "params": params[:80], "auroc enreg.": rec.get("auroc")}
        y_raw, s_raw = rec.get("y_true"), rec.get("y_scores")
        if y_raw is None or s_raw is None:
            row["statut"] = "pas de scores"
            rows.append(row)
            continue
        try:
            y = np.asarray(y_raw, dtype=int).ravel()
            s = as_scores(s_raw)
        except (ValueError, TypeError) as e:
            row["statut"] = f"scores illisibles : {e}"[:60]
            rows.append(row)
            continue
        if len(y) != len(s):
            row["statut"] = f"longueurs y/s incohérentes : {len(y)}/{len(s)}"
            rows.append(row)
            continue
        ids = rec.get("ids")
        if ids is None:
            fp = "sans ids"
        elif len(ids) != len(y):
            fp = f"ids ≠ n ({len(ids)})"
        else:
            fp = fingerprint(ids, y)
        ok = np.isfinite(s)
        row.update({"n": len(y), "n IA": int(y.sum()), "NaN": int((~ok).sum()), "empreinte": fp})
        if ok.any() and len(np.unique(y[ok])) == 2:
            row.update(score_metrics(y[ok], s[ok]))
            row["statut"] = "AUC < 0.5 : inversé ?" if row["AUC"] < 0.5 else "ok"
        else:
            row["statut"] = "mono-classe"
        rows.append(row)

inv = pd.DataFrame(rows)
inv["runs du hash"] = inv.groupby("hash")["ligne"].transform("count")
if "AUC" in inv:
    inv["écart AUC du hash"] = inv.groupby("hash")["AUC"].transform(lambda a: a.max() - a.min())

bad = pd.DataFrame(corrupt)
if not bad.empty:
    bad["dernière ligne"] = bad["ligne"] == n_lines

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None,
                       "display.max_colwidth", 80):
    print(f"=== 0. Lignes illisibles : {len(bad)} sur {n_lines} ===")
    if not bad.empty:
        print(bad.to_string(index=False))

    if "empreinte" in inv:
        print("\n=== 1. Ensembles évalués (une empreinte = un même ensemble de reviews) ===")
        groups = (inv.groupby(["empreinte", "split", "subset", "seed ds"], dropna=False)
                     .agg(n=("n", "first"), runs=("ligne", "count"),
                          détecteurs=("détecteur", lambda x: sorted({str(v) for v in x}))))
        print(groups.to_string())

    print("\n=== 2. Inventaire des runs ===")
    cols = ["ligne", "hash", "détecteur", "type", "params", "empreinte", "n", "n IA", "NaN", "AUC", "auroc enreg.",
            "TPR@1%", "TPR@5%", "acc max", "bacc max", "runs du hash", "écart AUC du hash", "statut"]
    print(inv.sort_values(["empreinte", "détecteur", "ligne"])[[c for c in cols if c in inv]]
             .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

=== 0. Lignes illisibles : 1 sur 42 ===
 ligne  taille (car.)  pos. erreur  finit par \n                          erreur         hash detector.name dataset.split  dernière ligne
    42        5148405      5145242         False Unterminated string starting at 5a3272825497     binocular          test            True

=== 1. Ensembles évalués (une empreinte = un même ensemble de reviews) ===
                                       n  runs                                                             détecteurs
empreinte  split subset  seed ds                                                                                     
0fa5e73f02 test  600.0   42        568.0     2                                                       [cristal_anchor]
3aa6526c21 test  10.0    42         10.0     4                                                       [cristal_anchor]
3da4687b23 test  10000.0 42       9424.0     4                                                                 [sage]
4149f0e990 test  N

In [3]:
# notebook : detectors_comparison.ipynb — cellule 2 : bloc A (100 reviews) — Binoculars, DetectGPT, SAGE
# à exécuter après la cellule 1 (réutilise PATH, as_scores, score_metrics)
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

LINES = {"Binoculars": 7, "DetectGPT": 4, "SAGE réf. (580ae5ec)": 26, "SAGE 100 (l.13)": 13, "SAGE 100 (l.14)": 14}
ORIENT = {"Binoculars": -1}   # Hans et al. (2024) : score bas = texte machine ; autres détecteurs : score haut = IA
N_BOOT = 1000


def read_lines(path, wanted: set[int]) -> dict[int, dict]:
    out = {}
    with open(path) as f:
        for no, line in enumerate(f, start=1):
            if no in wanted:
                out[no] = json.loads(line)
                if len(out) == len(wanted):
                    break
    return out


def digest(text) -> str:
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()


def auc_ci(y: np.ndarray, s: np.ndarray, seed: int = 0) -> tuple[float, float]:
    """IC95 percentile par bootstrap simple (pas de grappes : identifiants absents des runs à 100 reviews)."""
    rng = np.random.default_rng(seed)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.integers(0, len(y), len(y))
        if 0 < y[idx].sum() < len(idx):
            vals.append(roc_auc_score(y[idx], s[idx]))
    return tuple(np.percentile(vals, [2.5, 97.5]))


recs = read_lines(PATH, set(LINES.values()))
runs = {}
for name, no in LINES.items():
    r = recs[no]
    y = np.asarray(r["y_true"], dtype=int).ravel()
    s = as_scores(r["y_scores"]) * ORIENT.get(name, 1)
    content = r.get("content")
    runs[name] = {"y": y, "s": s, "dig": [digest(c) for c in content] if content is not None else None}
    print(f"{name:<22} ligne {no:>2} | n {len(y)} | IA {int(y.sum())} | textes enregistrés : {content is not None}")

# ------------------------------------------------------------------ 1. Binoculars et DetectGPT : même ensemble ?
b, d = runs["Binoculars"], runs["DetectGPT"]
if b["dig"] is not None and d["dig"] is not None:
    same = sorted(zip(b["dig"], b["y"])) == sorted(zip(d["dig"], d["y"]))
    print(f"\nBinoculars et DetectGPT sur le même ensemble (textes + labels) : {same}")
else:
    print("\ntextes absents : identité des ensembles non vérifiable (même configuration de dataset seulement)")

# ------------------------------------------------------------------ 2. SAGE de référence sur ces 100 reviews
ref = runs["SAGE réf. (580ae5ec)"]
block = {k: runs[k] for k in ("Binoculars", "DetectGPT")}
if b["dig"] is not None and ref["dig"] is not None:
    table = pd.DataFrame({"dig": ref["dig"], "y": ref["y"], "s": ref["s"]})
    conflicts = table.groupby("dig")["s"].nunique().gt(1).sum()
    lookup = table.drop_duplicates("dig").set_index("dig")
    found = [g in lookup.index for g in b["dig"]]
    print(f"reviews du bloc A retrouvées dans le run de référence : {sum(found)} / {len(found)} "
          f"| textes en double à scores différents : {conflicts}")
    if all(found):
        y_ref = lookup.loc[b["dig"], "y"].to_numpy(dtype=int)
        if (y_ref == b["y"]).all():
            block["SAGE réf. (580ae5ec)"] = {"y": b["y"], "s": lookup.loc[b["dig"], "s"].to_numpy(dtype=float)}
        else:
            print("⚠️ labels différents entre le bloc A et le run de référence : appariement rejeté")
for k in ("SAGE 100 (l.13)", "SAGE 100 (l.14)"):
    block[k] = runs[k]

# ------------------------------------------------------------------ 3. métriques du bloc A
rows = {}
for name, r in block.items():
    m = score_metrics(r["y"], r["s"])
    lo, hi = auc_ci(r["y"], r["s"])
    rows[name] = {**m, "AUC IC bas": lo, "AUC IC haut": hi}
print("\n=== Bloc A : 100 reviews (TPR@1% non estimable avec 50 humaines) ===")
print(pd.DataFrame(rows).T.to_string(float_format=lambda x: f"{x:.4f}"))


Binoculars             ligne  7 | n 100 | IA 50 | textes enregistrés : False
DetectGPT              ligne  4 | n 100 | IA 50 | textes enregistrés : False
SAGE réf. (580ae5ec)   ligne 26 | n 9424 | IA 5000 | textes enregistrés : True
SAGE 100 (l.13)        ligne 13 | n 100 | IA 50 | textes enregistrés : False
SAGE 100 (l.14)        ligne 14 | n 100 | IA 50 | textes enregistrés : False

textes absents : identité des ensembles non vérifiable (même configuration de dataset seulement)

=== Bloc A : 100 reviews (TPR@1% non estimable avec 50 humaines) ===
                   AUC  TPR@1%  TPR@5%  acc max  bacc max  AUC IC bas  AUC IC haut
Binoculars      0.6908  0.2400  0.3600   0.6600    0.6600      0.5781       0.7920
DetectGPT       0.8208  0.2800  0.4000   0.7800    0.7800      0.7360       0.8912
SAGE 100 (l.13) 0.9252  0.3600  0.5400   0.8600    0.8600      0.8663       0.9672
SAGE 100 (l.14) 0.8988  0.3200  0.3600   0.8600    0.8600      0.8279       0.9527


In [4]:
# notebook : detectors_comparison.ipynb — cellule 3 : configurations SAGE comparées (remplace la section 4 de la cellule 2)
import pandas as pd

keys = sorted({k for no in (26, 13, 14) for k in recs[no] if k.startswith("training.")})
cfg = pd.DataFrame({f"ligne {no}": {k: repr(recs[no].get(k)) for k in keys} for no in (26, 13, 14)})
print("=== Configurations d'entraînement SAGE (lignes 26, 13, 14) : clés qui diffèrent ===")
print(cfg[cfg.nunique(axis=1) > 1].to_string() if len(cfg) else "aucune clé training.*")

=== Configurations d'entraînement SAGE (lignes 26, 13, 14) : clés qui diffèrent ===
                                                                                           ligne 26 ligne 13 ligne 14
training.model.batch_size                                                                        64     None     None
training.model.feature_version.segmentation                                                       2     None     None
training.model.feature_version.subj_category  ['Assertion', 'Expressive', 'Question', 'Suggestion']     None     None
training.model.method                                                                    'training'     None     None
training.model.name                                                                    'sage_gru_1'     None     None
training.model.params_model.hidden                                                               32     None     None
training.model.params_model.n_features                                                    

In [7]:
# notebook : detectors_comparison.ipynb — cellule 4 : valeurs du tableau récapitulatif (AUC, Acc max, TPR@1 %, TPR@0,1 %)
# à exécuter après les cellules 1 et 2 (réutilise PATH, as_scores, read_lines, digest)
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

RECAP_SCORES_B = Path(os.environ["CACHE_DIR"]) / "recap" / "scores_partie_B.parquet"   # export cellule 16 du récapitulatif
JSONL_RUNS = {  # nom -> (ligne de results.jsonl, orientation : +1 = score haut pour l'IA, -1 = score bas, ensemble)
    "Binoculars": (7, -1, "100 reviews"),                        # Hans et al. (2024)
    "Fast-DetectGPT": (5, +1, "100 reviews"),                    # Bao et al. (2024)
    "SAGE-APRDB (réf. 580ae5ec)": (26, +1, "test 9 424"),
}
TARGETS = {"TPR@1%": 0.01, "TPR@0.1%": 0.001}
REPORTED_B = {"Sem-Detect E3 (a)": (0.9801, 0.6999), "SAGE-APRDB": (0.9590, 0.5161)}   # tab:cross_dataset


def summary(y: np.ndarray, s: np.ndarray) -> dict:
    """AUC, accuracy maximale sur la ROC (seuil oracle), TPR interpolés et faux positifs attendus à chaque FPR cible."""
    fpr, tpr, _ = roc_curve(y, s, drop_intermediate=False)
    pos = int(y.sum())
    neg = len(y) - pos
    acc = (tpr * pos + (1 - fpr) * neg) / (pos + neg)
    row = {"n": len(y), "n IA": pos, "n hum.": neg,
           "AUC": float(roc_auc_score(y, s)), "acc max": float(acc.max())}
    for name, t in TARGETS.items():
        row[name] = float(np.interp(t, fpr, tpr))
        row[f"FP att. {name[4:]}"] = t * neg   # < 1 : TPR non estimable à ce FPR
    return row


rows, data = {}, {}

# ------------------------------------------------------------------ 1. runs de results.jsonl
recs4 = read_lines(PATH, {no for no, _, _ in JSONL_RUNS.values()})
for name, (no, sign, ens) in JSONL_RUNS.items():
    r = recs4[no]
    y = np.asarray(r["y_true"], dtype=int).ravel()
    s = as_scores(r["y_scores"]) * sign
    dig = [digest(c) for c in r["content"]] if r.get("content") is not None else None
    data[name] = (y, s, dig)
    rows[(ens, name)] = summary(y, s)

# ------------------------------------------------------------------ 2. SAGE de référence sur les 100 reviews des baselines
by_, bs_, bd_ = data["Binoculars"]
fy_, fs_, fd_ = data["Fast-DetectGPT"]
ry_, rs_, rd_ = data["SAGE-APRDB (réf. 580ae5ec)"]
if bd_ is not None and fd_ is not None:
    print(f"Binoculars et Fast-DetectGPT sur le même ensemble : {sorted(zip(bd_, by_)) == sorted(zip(fd_, fy_))}")
if fd_ is not None and rd_ is not None:
    lk = pd.DataFrame({"d": rd_, "y": ry_, "s": rs_}).drop_duplicates("d").set_index("d")
    hit = [d in lk.index for d in fd_]
    print(f"reviews des baselines retrouvées dans le run SAGE de référence : {sum(hit)} / {len(hit)}")
    if all(hit) and (lk.loc[fd_, "y"].to_numpy(dtype=int) == fy_).all():
        rows[("100 reviews", "SAGE-APRDB (réf.) sur les mêmes reviews")] = summary(fy_, lk.loc[fd_, "s"].to_numpy(float))
    else:
        print("appariement impossible : pas de ligne SAGE sur les 100 reviews")
else:
    print("textes absents d'un des runs : appariement SAGE / baselines impossible")

# ------------------------------------------------------------------ 3. pilote APRDB : Sem-Detect E3 (a) et SAGE-APRDB
if RECAP_SCORES_B.exists():
    sb = pd.read_parquet(RECAP_SCORES_B)
    sb = sb[sb["commun"]]
    y = sb["class"].eq("ai").to_numpy(dtype=int)
    for col, (auc_ref, tpr_ref) in REPORTED_B.items():
        r = summary(y, sb[col].to_numpy(dtype=float))
        ok = abs(r["AUC"] - auc_ref) < 5e-4 and abs(r["TPR@1%"] - tpr_ref) < 5e-4
        print(f"{'✅' if ok else '⚠️'} {col} : AUC {r['AUC']:.4f} (rapporté {auc_ref}) | "
              f"TPR@1% {r['TPR@1%']:.4f} (rapporté {tpr_ref})")
        rows[("pilote APRDB 1 841", col)] = r
else:
    print(f"{RECAP_SCORES_B} introuvable : vérifier RECAP_DIR dans la cellule 1 du notebook récapitulatif")

# ------------------------------------------------------------------ 4. tableau
out = pd.DataFrame(rows).T
out.index.names = ["ensemble", "détecteur"]
cols = ["n", "n IA", "n hum.", "AUC", "acc max", "TPR@1%", "FP att. 1%", "TPR@0.1%", "FP att. 0.1%"]
print("\n=== Valeurs du tableau récapitulatif ===")
print(out[cols].to_string(float_format=lambda x: f"{x:.4f}"))

textes absents d'un des runs : appariement SAGE / baselines impossible
/data2/cyaiche/cache_dir/recap/scores_partie_B.parquet introuvable : vérifier RECAP_DIR dans la cellule 1 du notebook récapitulatif

=== Valeurs du tableau récapitulatif ===
                                               n      n IA    n hum.    AUC  acc max  TPR@1%  FP att. 1%  TPR@0.1%  FP att. 0.1%
ensemble    détecteur                                                                                                           
100 reviews Binoculars                  100.0000   50.0000   50.0000 0.6908   0.6600  0.2400      0.5000    0.2400        0.0500
            Fast-DetectGPT              100.0000   50.0000   50.0000 0.7592   0.7900  0.4400      0.5000    0.4400        0.0500
test 9 424  SAGE-APRDB (réf. 580ae5ec) 9424.0000 5000.0000 4424.0000 0.9642   0.9012  0.5464     44.2400    0.2518        4.4240


In [8]:
# notebook : detectors_comparison.ipynb — cellule 7 (remplace la précédente) : lecture tolérante + valeurs Binoculars / Fast-DetectGPT
# à exécuter après les cellules 1, 2 et 4 (réutilise PATH, as_scores, fingerprint, summary)
import json
import re

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

REF_FINGERPRINT = "3da4687b23"   # ensemble de référence de SAGE (cellule 1)
ZERO_SHOT = {"binocular": -1, "fast_detect_gpt": +1}   # orientation a priori : -1 = score bas pour l'IA (Hans et al., 2024)
REPORTED = {"binocular": (0.9836, 0.7676), "fast_detect_gpt": (0.7304, 0.3974)}   # logs de l'Evaluator : AUROC, TPR@1 %

DECODER = json.JSONDecoder()
RECORD_START = re.compile(r'\{"hash":\s*"')
DETECTOR_NAME = re.compile(r'"detector\.name":\s*"([^"]+)"')
DAMAGED = []


def iter_records(path):
    """Lit results.jsonl ligne à ligne ; dans une ligne corrompue, récupère chaque objet complet commençant par {"hash": …}."""
    with open(path) as f:
        for no, line in enumerate(f, start=1):
            try:
                yield no, json.loads(line)
                continue
            except json.JSONDecodeError as err:
                error = str(err)
            recovered = []
            for m in RECORD_START.finditer(line):
                try:
                    obj, _ = DECODER.raw_decode(line, m.start())
                except json.JSONDecodeError:
                    continue
                recovered.append(obj.get("detector.name"))
                yield no, obj
            DAMAGED.append({"ligne": no, "longueur": len(line), "erreur": error[:70],
                            "débuts d'enregistrement": len(RECORD_START.findall(line)),
                            "détecteurs mentionnés": sorted(set(DETECTOR_NAME.findall(line))),
                            "récupérés": recovered})


last = {}
for no, rec in iter_records(PATH):
    d = rec.get("detector.name")
    if d in ZERO_SHOT and rec.get("dataset.subset_size") == 10000:
        last[d] = (no, rec)   # garde le run le plus récent

print("=== Lignes corrompues ===")
print(pd.DataFrame(DAMAGED).to_string(index=False) if DAMAGED else "aucune")

rows = {}
for d, sign in ZERO_SHOT.items():
    if d not in last:
        print(f"\n⚠️ aucun enregistrement {d} valide sur un subset de 10 000")
        continue
    no, rec = last[d]
    y = np.asarray(rec["y_true"], dtype=int).ravel()
    raw = as_scores(rec["y_scores"])
    fp = fingerprint(rec["ids"], y) if rec.get("ids") is not None else "sans ids"
    m = summary(y, sign * raw)
    auc_ref, tpr_ref = REPORTED[d]
    ok = abs(m["AUC"] - auc_ref) < 5e-4 and abs(m["TPR@1%"] - tpr_ref) < 5e-4
    params = {k.split("detector.params.", 1)[1]: v for k, v in rec.items()
              if k.startswith("detector.params.") and "device" not in k}
    print(f"\n{d} | ligne {no} | n {len(y)} | IA {int(y.sum())} | empreinte {fp} "
          f"{'✅ = référence' if fp == REF_FINGERPRINT else '⚠️ ≠ ' + REF_FINGERPRINT}")
    print(f"  paramètres : {params}")
    print(f"  AUC score brut {roc_auc_score(y, raw):.4f} | orienté {m['AUC']:.4f} | "
          f"{'✅' if ok else '⚠️'} logs : AUROC {auc_ref}, TPR@1% {tpr_ref} -> recalculé {m['TPR@1%']:.4f}")
    rows[d] = m

if rows:
    print("\n=== Valeurs pour tab:detector_comparison ===")
    print(pd.DataFrame(rows).T[["n", "n IA", "n hum.", "AUC", "acc max", "TPR@1%", "TPR@0.1%", "FP att. 0.1%"]]
          .to_string(float_format=lambda x: f"{x:.4f}"))

=== Lignes corrompues ===
 ligne  longueur                                                        erreur  débuts d'enregistrement        détecteurs mentionnés         récupérés
    42  29869732 Expecting ',' delimiter: line 1 column 5148408 (char 5148407)                        2 [binocular, fast_detect_gpt] [fast_detect_gpt]

⚠️ aucun enregistrement binocular valide sur un subset de 10 000

fast_detect_gpt | ligne 42 | n 9468 | IA 5000 | empreinte b2a9e5251c ⚠️ ≠ 3da4687b23
  paramètres : {'scoring_model_name': 'gpt2-xl', 'sampling_model_name': 'gpt2-xl', 'discrepancy_analytic': 'True'}
  AUC score brut 0.7304 | orienté 0.7304 | ✅ logs : AUROC 0.7304, TPR@1% 0.3974 -> recalculé 0.3974

=== Valeurs pour tab:detector_comparison ===
                        n      n IA    n hum.    AUC  acc max  TPR@1%  TPR@0.1%  FP att. 0.1%
fast_detect_gpt 9468.0000 5000.0000 4468.0000 0.7304   0.7214  0.3974    0.3068        4.4680
